# 10. Artifact naming and stage metadata

The bookkeeping layer of the pipeline turns configuration into deterministic artifact paths and human-readable metadata. This notebook exercises the real `ArtifactRegistry` (filename and path construction from crop and stack tags) and `MetadataManager` (tomogram and inputs metadata dictionaries), and shows how the crop region and stack identifiers propagate into the generated tags. Only in-memory objects are used; no artifacts are written.

**Modules exercised:** pipelines.processing_pipeline.artifacts (ArtifactRegistry, MetadataManager), configuration.processing_config (ProcessingConfiguration tags), tools.data.regions (CropRegion)

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## Build a configuration

The artifact tags derive from the crop identifier string and the stack identifiers, so two different crops produce two different filename sets.

In [ ]:
from configuration.processing_config import (
    ProcessingConfiguration, TomogramConfiguration, PathConfiguration,
)
from tools.data.regions import CropRegion
from pipelines.processing_pipeline.artifacts import ArtifactRegistry, MetadataManager
from tools.monitoring.logger import Logger

logger = Logger(log_dir='/tmp/nb10_logs', name='nb10', level='WARNING')
crop   = CropRegion(azimuth_start=1000, azimuth_end=2000, range_start=500, range_end=1500)
config = ProcessingConfiguration(
    crop                     = crop,
    paths                    = PathConfiguration(main_directory=Path('/tmp/nb10_run')),
    full_stack_identifier    = 'flaca',
    reduced_stack_identifier = 'flaca_2',
)
print('tomogram_tag :', config.tomogram_tag)
print('parameter_tag:', config.parameter_tag)

## Artifact filenames and paths

`ArtifactRegistry.artifact_filenames` returns the seven artifact names; `artifact_path` resolves each under the run data directory.

In [ ]:
registry  = ArtifactRegistry(config, logger)
filenames = registry.artifact_filenames()
for key, name in filenames.items():
    print(f'  {key:24s} -> {name}')

print()
print('tomogram_reduced path:')
print('  ', registry.artifact_path('tomogram_reduced'))

## Tag sensitivity to the crop

Changing the crop changes the identifier string and therefore every filename. We build a second config with a different crop and compare the reduced-tomogram filenames.

In [ ]:
crop_b   = CropRegion(azimuth_start=3000, azimuth_end=3500, range_start=200, range_end=800)
config_b = ProcessingConfiguration(crop=crop_b, paths=PathConfiguration(main_directory=Path('/tmp/nb10_run_b')))
reg_b    = ArtifactRegistry(config_b, logger)

print('crop A identifier:', crop.as_identifier_string())
print('crop B identifier:', crop_b.as_identifier_string())
print('A tomogram_reduced:', registry.artifact_filenames()['tomogram_reduced'])
print('B tomogram_reduced:', reg_b.artifact_filenames()['tomogram_reduced'])

## Stage metadata dictionaries

`MetadataManager.build_tomogram_metadata` and `build_inputs_metadata` assemble the key-value records written to the `meta` directory. We construct them in memory and tabulate the entries.

In [ ]:
manager = MetadataManager(config, logger)

tomo_meta = manager.build_tomogram_metadata(
    variant          = 'reduced',
    output_path      = registry.artifact_path('tomogram_reduced'),
    stack_identifier = config.reduced_stack_identifier,
    cfg              = config.input_configs,
)
print('tomogram metadata entries:')
for k, v in tomo_meta.items():
    print(f'  {k:14s}: {v}')

In [ ]:
inputs_meta = manager.build_inputs_metadata(
    primary_path         = registry.artifact_path('primary_reduced'),
    secondaries_path     = registry.artifact_path('secondaries_reduced'),
    interferograms_path  = registry.artifact_path('interferograms_reduced'),
    primary_shape        = (480, 1000),
    secondaries_shape    = (5, 480, 1000),
    interferograms_shape = (5, 480, 1000),
)
print('inputs metadata entries:')
for k, v in inputs_meta.items():
    print(f'  {k:20s}: {v}')

## Visual summary of the metadata records

A simple table figure makes the propagation of crop and stack identifiers into the metadata visible at a glance.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.axis('off')
rows = list(tomo_meta.items())
table = ax.table(cellText=[[k, str(v)] for k, v in rows], colLabels=['key', 'value'], loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.0, 1.25)
ax.set_title('tomogram_reduced stage metadata')
fig.tight_layout()
plt.show()

## Expected visual outcome

The two crops should yield distinct reduced-tomogram filenames differing only in the crop identifier substring. The metadata tables should list the crop, stack identifiers, polarisation, filter and beamforming method consistent with the configuration, confirming the bookkeeping layer derives all artifact names and metadata deterministically from the config.